# Taazi Khabar — QLoRA Fine-Tuning (Kaggle)

Fine-tune a single LoRA adapter on all 3 UPSC tasks (summarizer, article filter, quiz setter).
The base model learns to switch behavior based on the `instruction` field in each example.

**Setup:**
1. Create a Kaggle Dataset with `combined.jsonl` at `kaggle/datasets/<your-user>/taazi-training-data/`
2. Or upload `combined.jsonl` to the notebook via "Add Input" → Upload
3. Set `DATASET_PATH` below to match
4. Settings → Accelerator → **T4 GPU x2** (or P100)
5. Run all cells

**Tasks in the dataset:**
- `Summarize this UPSC article...` → structured GK summary (298 examples)
- `Determine if this is UPSC-relevant...` → YES/NO (311 examples)
- `Generate UPSC-style MCQ...` → MCQ with 4 options (95 examples)

**Output:** `/kaggle/working/taazi-adapter/final/` — download after training.

In [ ]:
# === CONFIGURATION ===

# Path to combined.jsonl (upload to Kaggle Dataset or use Add Input)
DATASET_PATH = "/kaggle/input/taazi-training-data/combined.jsonl"

# Output directory (saved to Kaggle working dir)
OUTPUT_DIR = "/kaggle/working/taazi-adapter"

# Model
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"  # 3.8B

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2
MAX_SEQ_LENGTH = 2048
WARMUP_STEPS = 20

## 1. Install Dependencies

In [ ]:
!pip install -q datasets accelerate peft bitsandbytes transformers trl
import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
import json
from datasets import Dataset

records = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

texts = []
for r in records:
    t = f"<|user|>\n{r['instruction']}\n{r['input']}<|end|>\n<|assistant|>\n{r['output']}<|endoftext|>"
    texts.append(t)

dataset = Dataset.from_dict({"text": texts})
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Loaded {len(records)} records")
print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

## 3. Load Tokenizer + Quantized Model

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    ddp_find_unused_parameters=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

trainer.train()

## 5. Save Adapter

In [ ]:
import os
import zipfile

adapter_path = os.path.join(OUTPUT_DIR, "final")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")
print("Files:", os.listdir(adapter_path))

# Zip for easy download
zip_path = "/kaggle/working/taazi-adapter-final.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(adapter_path):
        for f in files:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, os.path.dirname(adapter_path))
            zf.write(fp, arcname)

print(f"\nZip created: {zip_path}")
print("Download via Kaggle sidebar → Output → ... → Download all")

## 6. Inference Test (Optional)

In [ ]:
from peft import PeftModel

# Reload base + adapter
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, adapter_path)

test_prompts = [
    "<|user|>\nSummarize this UPSC article and create a GK summary with Prelims, Mains, and Interview sections.\nThe Supreme Court has upheld the constitutional validity of the Places of Worship Act, 1991.<|end|>\n<|assistant|>",
    "<|user|>\nDetermine if this news is relevant for UPSC Civil Services Exam. Answer only YES or NO.\nLocal municipality inaugurates new park in residential colony.<|end|>\n<|assistant|>",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.2,
        do_sample=True,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("="*60)
    print(result)
    print()